In [1]:
import os
import glob
import gc
import numpy as np
import torch
from torch.utils.data import IterableDataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ==========================================
# 1. KAGGLE SPECIFIC PATHS
# ==========================================
# Input path from your Kaggle dataset (Note the double folder structure)
TOKENIZED_DIR = "/kaggle/input/datasets/tanishtushid/tokenized/tokenized"

# Output path (Kaggle saves files here so you can download them later)
MODELS_DIR = "/kaggle/working/models"
os.makedirs(MODELS_DIR, exist_ok=True)

print("Looking for data in:", TOKENIZED_DIR)

# ==========================================
# 2. MODEL CONFIGURATION
# ==========================================
MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "eng_Latn"
TGT_LANG = "hin_Deva"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Looking for data in: /kaggle/input/datasets/tanishtushid/tokenized/tokenized
Using device: cuda
GPU: Tesla T4


In [2]:
!pip install -q --upgrade --force-reinstall --no-cache-dir \
    "numpy>=2.1,<2.3" \
    "scipy>=1.14" \
    "scikit-learn>=1.6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 163.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 199.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 282.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 181.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 285.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.0/474.0 kB 278.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.18.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupy

In [3]:
import os

MODELS_DIR = "/kaggle/working/models"

os.makedirs(MODELS_DIR, exist_ok=True)

print("Models directory exists:", os.path.exists(MODELS_DIR))
print("Path:", MODELS_DIR)

Models directory exists: True
Path: /kaggle/working/models


In [4]:
# ==========================================
# 3. LOAD TOKENIZER AND DATASET
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, src_lang=SRC_LANG, tgt_lang=TGT_LANG
)

# Find all .npz files in your exact Kaggle path
train_files = sorted(glob.glob(os.path.join(TOKENIZED_DIR, "*.npz")))
print(f"\nFound {len(train_files)} .npz files.")

for f in train_files:
    print(os.path.basename(f))

# Split based on your 14 files (12 train, 1 val, 1 test)
# Adjust the numbers if you add more chunks later
train_chunk_files = train_files[:12]
validation_chunk_files = train_files[12:13]
test_chunk_files = train_files[13:14]

print(f"\nTraining on {len(train_chunk_files)} chunks")
print(f"Validating on {len(validation_chunk_files)} chunks")

# ==========================================
# 4. CUSTOM DATASET CLASS
# ==========================================
class NPZTokenizedDataset(IterableDataset):
    def __init__(self, files):
        self.files = sorted(files)

    def __iter__(self):
        for file in self.files:
            # Load the NPZ file
            with np.load(file) as data:
                encoder_inputs = data["encoder_inputs"]
                decoder_targets = data["decoder_targets"]

                for i in range(len(encoder_inputs)):
                    yield {
                        "encoder_inputs": torch.tensor(
                            encoder_inputs[i], dtype=torch.long
                        ),
                        "decoder_targets": torch.tensor(
                            decoder_targets[i], dtype=torch.long
                        )
                    }

# Initialize Datasets
train_dataset = NPZTokenizedDataset(train_chunk_files)
val_dataset = NPZTokenizedDataset(validation_chunk_files)

print("\nDatasets initialized successfully.")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]


Found 14 .npz files.
train_chunk_000.npz
train_chunk_001.npz
train_chunk_002.npz
train_chunk_003.npz
train_chunk_004.npz
train_chunk_005.npz
train_chunk_006.npz
train_chunk_007.npz
train_chunk_008.npz
train_chunk_009.npz
train_chunk_010.npz
train_chunk_011.npz
train_chunk_012.npz
train_chunk_013.npz

Training on 12 chunks
Validating on 1 chunks

Datasets initialized successfully.


In [5]:
!pip install -q --no-cache-dir "huggingface-hub>=1.23.0,<2.0"

In [6]:
!pip install -q --no-cache-dir transformers==4.57.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 228.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 305.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 152.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [7]:
!pip install -q --no-cache-dir --force-reinstall \
    "transformers==5.16.1" \
    "huggingface-hub>=1.0,<2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 231.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 241.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 352.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 271.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.2/111.2 kB 308.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.7/221.7 kB 325.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 215.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 252.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 208.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 305.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 278.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
import transformers
import huggingface_hub
import torch

print("Transformers:", transformers.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Transformers: 5.16.1
HuggingFace Hub: 1.29.0
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [9]:
!pip install -q --no-cache-dir bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 326.0 MB/s eta 0:00:00


In [10]:
import gc
import torch
import bitsandbytes as bnb
from transformers import AutoModelForSeq2SeqLM

# ==========================================
# 5. INITIALIZE MODEL AND OPTIMIZER
# ==========================================

gc.collect()
torch.cuda.empty_cache()

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# ==========================================
# LOAD NLLB MODEL
# ==========================================

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

print("Model loaded successfully!")

# ==========================================
# NLLB: FORCE HINDI OUTPUT
# ==========================================

forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

model.generation_config.forced_bos_token_id = forced_bos_token_id

print("Hindi token ID:", forced_bos_token_id)

# ==========================================
# MEMORY OPTIMIZATION
# ==========================================

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)

model.config.use_cache = False

# ==========================================
# 8-BIT OPTIMIZER
# ==========================================

optimizer = bnb.optim.AdamW8bit(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

# ==========================================
# MIXED PRECISION
# ==========================================

scaler = torch.amp.GradScaler("cuda")

# ==========================================
# TRAINING CONFIG
# ==========================================

EPOCHS = 5

BATCH_SIZE = 8

GRAD_ACCUM_STEPS = 32

MAX_STEPS = 2500

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("\nModel, Optimizer, and DataLoader are ready!")

print(
    f"Effective Batch Size: "
    f"{BATCH_SIZE * GRAD_ACCUM_STEPS}"
)

Device: cuda


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.46GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully!
Hindi token ID: 256068

Model, Optimizer, and DataLoader are ready!
Effective Batch Size: 256


In [11]:
# ==========================================
# 6. TRAINING LOOP
# ==========================================
model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    steps_done = 0
    optimizer.zero_grad(set_to_none=True)

    print(f"\n========== Epoch {epoch + 1}/{EPOCHS} ==========")

    for step, batch in enumerate(train_loader):
        if step >= MAX_STEPS:
            break

        # Move batch to GPU
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        # Mixed Precision Forward Pass
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(
                input_ids=batch["encoder_inputs"],
                labels=batch["decoder_targets"]
            )
            # Normalize loss for gradient accumulation
            loss = outputs.loss / GRAD_ACCUM_STEPS

        # Backward Pass
        scaler.scale(loss).backward()

        # Gradient Accumulation Step
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * GRAD_ACCUM_STEPS
        steps_done += 1

        if step % 100 == 0:
            print(f"Epoch {epoch+1} | Step {step+1}/{MAX_STEPS} | Loss: {loss.item() * GRAD_ACCUM_STEPS:.4f}")

    # Save Checkpoint at the end of every epoch
    save_path = os.path.join(MODELS_DIR, f"nllb_epoch_{epoch+1}")
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Epoch {epoch+1} finished. Model saved to {save_path}")
    
    # Clean up memory
    torch.cuda.empty_cache()
    gc.collect()


========== Epoch 1/5 ==========
Epoch 1 | Step 1/2500 | Loss: 13.0712
Epoch 1 | Step 101/2500 | Loss: 12.7693
Epoch 1 | Step 201/2500 | Loss: 11.7369
Epoch 1 | Step 301/2500 | Loss: 12.0674
Epoch 1 | Step 401/2500 | Loss: 11.0848
Epoch 1 | Step 501/2500 | Loss: 10.4567
Epoch 1 | Step 601/2500 | Loss: 9.8458
Epoch 1 | Step 701/2500 | Loss: 9.2875
Epoch 1 | Step 801/2500 | Loss: 8.3569
Epoch 1 | Step 901/2500 | Loss: 8.5001
Epoch 1 | Step 1001/2500 | Loss: 9.4778
Epoch 1 | Step 1101/2500 | Loss: 7.1467
Epoch 1 | Step 1201/2500 | Loss: 6.8540
Epoch 1 | Step 1301/2500 | Loss: 6.9708
Epoch 1 | Step 1401/2500 | Loss: 7.2343
Epoch 1 | Step 1501/2500 | Loss: 6.2668
Epoch 1 | Step 1601/2500 | Loss: 6.1305
Epoch 1 | Step 1701/2500 | Loss: 6.9203
Epoch 1 | Step 1801/2500 | Loss: 5.4954
Epoch 1 | Step 1901/2500 | Loss: 6.4440
Epoch 1 | Step 2001/2500 | Loss: 5.8914
Epoch 1 | Step 2101/2500 | Loss: 5.3377
Epoch 1 | Step 2201/2500 | Loss: 6.0174
Epoch 1 | Step 2301/2500 | Loss: 5.3191
Epoch 1 | Ste

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1 finished. Model saved to /kaggle/working/models/nllb_epoch_1

========== Epoch 2/5 ==========
Epoch 2 | Step 1/2500 | Loss: 5.6470
Epoch 2 | Step 101/2500 | Loss: 4.6302
Epoch 2 | Step 201/2500 | Loss: 4.6789
Epoch 2 | Step 301/2500 | Loss: 5.7053
Epoch 2 | Step 401/2500 | Loss: 5.1053
Epoch 2 | Step 501/2500 | Loss: 4.3726
Epoch 2 | Step 601/2500 | Loss: 4.3257
Epoch 2 | Step 701/2500 | Loss: 4.3596
Epoch 2 | Step 801/2500 | Loss: 4.0469
Epoch 2 | Step 901/2500 | Loss: 4.8201
Epoch 2 | Step 1001/2500 | Loss: 6.3930
Epoch 2 | Step 1101/2500 | Loss: 3.9830
Epoch 2 | Step 1201/2500 | Loss: 3.9415
Epoch 2 | Step 1301/2500 | Loss: 4.2037
Epoch 2 | Step 1401/2500 | Loss: 4.7278
Epoch 2 | Step 1501/2500 | Loss: 3.8960
Epoch 2 | Step 1601/2500 | Loss: 3.8944
Epoch 2 | Step 1701/2500 | Loss: 4.8103
Epoch 2 | Step 1801/2500 | Loss: 3.5488
Epoch 2 | Step 1901/2500 | Loss: 4.5511
Epoch 2 | Step 2001/2500 | Loss: 3.9844
Epoch 2 | Step 2101/2500 | Loss: 3.4980
Epoch 2 | Step 2201/2500 | Los

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 2 finished. Model saved to /kaggle/working/models/nllb_epoch_2

========== Epoch 3/5 ==========
Epoch 3 | Step 1/2500 | Loss: 4.1262
Epoch 3 | Step 101/2500 | Loss: 3.1192
Epoch 3 | Step 201/2500 | Loss: 3.1956
Epoch 3 | Step 301/2500 | Loss: 4.1547
Epoch 3 | Step 401/2500 | Loss: 3.6095
Epoch 3 | Step 501/2500 | Loss: 2.9427
Epoch 3 | Step 601/2500 | Loss: 2.8703
Epoch 3 | Step 701/2500 | Loss: 2.9896
Epoch 3 | Step 801/2500 | Loss: 2.6996
Epoch 3 | Step 901/2500 | Loss: 3.4503
Epoch 3 | Step 1001/2500 | Loss: 5.1365
Epoch 3 | Step 1101/2500 | Loss: 2.6838
Epoch 3 | Step 1201/2500 | Loss: 2.6492
Epoch 3 | Step 1301/2500 | Loss: 2.9457
Epoch 3 | Step 1401/2500 | Loss: 3.5063
Epoch 3 | Step 1501/2500 | Loss: 2.6702
Epoch 3 | Step 1601/2500 | Loss: 2.6496
Epoch 3 | Step 1701/2500 | Loss: 3.5853
Epoch 3 | Step 1801/2500 | Loss: 2.3675
Epoch 3 | Step 1901/2500 | Loss: 3.4441
Epoch 3 | Step 2001/2500 | Loss: 2.8861
Epoch 3 | Step 2101/2500 | Loss: 2.3400
Epoch 3 | Step 2201/2500 | Los

...

Epoch 3 finished. Model saved to /kaggle/working/models/nllb_epoch_3

========== Epoch 4/5 ==========
Epoch 4 | Step 1/2500 | Loss: 3.1721
Epoch 4 | Step 101/2500 | Loss: 2.0335
Epoch 4 | Step 201/2500 | Loss: 2.1676
Epoch 4 | Step 301/2500 | Loss: 3.1084
Epoch 4 | Step 401/2500 | Loss: 2.5915
Epoch 4 | Step 501/2500 | Loss: 1.9018
Epoch 4 | Step 601/2500 | Loss: 1.8638
Epoch 4 | Step 701/2500 | Loss: 2.0169
Epoch 4 | Step 801/2500 | Loss: 1.7213
Epoch 4 | Step 901/2500 | Loss: 2.4731
Epoch 4 | Step 1001/2500 | Loss: 4.3039
Epoch 4 | Step 1101/2500 | Loss: 1.7270
Epoch 4 | Step 1201/2500 | Loss: 1.7145
Epoch 4 | Step 1301/2500 | Loss: 2.0109
Epoch 4 | Step 1401/2500 | Loss: 2.6856
Epoch 4 | Step 1501/2500 | Loss: 1.7875
Epoch 4 | Step 1601/2500 | Loss: 1.7643
Epoch 4 | Step 1701/2500 | Loss: 2.7482
Epoch 4 | Step 1801/2500 | Loss: 1.5079
Epoch 4 | Step 1901/2500 | Loss: 2.6398
Epoch 4 | Step 2001/2500 | Loss: 2.1017
Epoch 4 | Step 2101/2500 | Loss: 1.5351
Epoch 4 | Step 2201/2500 | Los

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 4 finished. Model saved to /kaggle/working/models/nllb_epoch_4

========== Epoch 5/5 ==========
Epoch 5 | Step 1/2500 | Loss: 2.4746
Epoch 5 | Step 101/2500 | Loss: 1.2685
Epoch 5 | Step 201/2500 | Loss: 1.4549
Epoch 5 | Step 301/2500 | Loss: 2.3855
Epoch 5 | Step 401/2500 | Loss: 1.8825
Epoch 5 | Step 501/2500 | Loss: 1.2173
Epoch 5 | Step 601/2500 | Loss: 1.1830
Epoch 5 | Step 701/2500 | Loss: 1.3698
Epoch 5 | Step 801/2500 | Loss: 1.0758
Epoch 5 | Step 901/2500 | Loss: 1.8523
Epoch 5 | Step 1001/2500 | Loss: 3.7380
Epoch 5 | Step 1101/2500 | Loss: 1.1147
Epoch 5 | Step 1201/2500 | Loss: 1.1266
Epoch 5 | Step 1301/2500 | Loss: 1.4536
Epoch 5 | Step 1401/2500 | Loss: 2.1519
Epoch 5 | Step 1501/2500 | Loss: 1.2462
Epoch 5 | Step 1601/2500 | Loss: 1.2475
Epoch 5 | Step 1701/2500 | Loss: 2.2162
Epoch 5 | Step 1801/2500 | Loss: 1.0036
Epoch 5 | Step 1901/2500 | Loss: 2.1477
Epoch 5 | Step 2001/2500 | Loss: 1.6309
Epoch 5 | Step 2101/2500 | Loss: 1.0674
Epoch 5 | Step 2201/2500 | Los

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 5 finished. Model saved to /kaggle/working/models/nllb_epoch_5


In [12]:
# ==========================================
# VALIDATION DATALOADER
# ==========================================

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Validation loader created.")

Validation loader created.


In [13]:
# ==========================================
# VALIDATE MODEL
# ==========================================

model.eval()

total_val_loss = 0.0
val_steps = 0

with torch.no_grad():

    for step, batch in enumerate(val_loader):

        # Move batch to GPU
        batch = {
            k: v.to(device, non_blocking=True)
            for k, v in batch.items()
        }

        # Mixed Precision
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(
                input_ids=batch["encoder_inputs"],
                labels=batch["decoder_targets"]
            )

        total_val_loss += outputs.loss.item()
        val_steps += 1

        # Validate maximum 500 batches
        if val_steps >= 500:
            break

val_loss = total_val_loss / val_steps

print("\n" + "=" * 50)
print(f"Validation Steps : {val_steps}")
print(f"Validation Loss  : {val_loss:.4f}")
print("=" * 50)

# Return model to training mode
model.train()


Validation Steps : 500
Validation Loss  : 3.4351


M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [14]:
# ==========================================
# TEST TRANSLATION
# ==========================================

model.eval()

sentences = [
    "How are you?",
    "What is your name?",
    "I am learning machine learning.",
    "I love programming.",
    "Where do you live?"
]

forced_bos_token_id = tokenizer.convert_tokens_to_ids("hin_Deva")

for text in sentences:

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_length=128,
            num_beams=4
        )

    translation = tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )[0]

    print(f"\nEnglish : {text}")
    print(f"Hindi   : {translation}")


English : How are you?
Hindi   : तुम कैसे हो?

English : What is your name?
Hindi   : आपका नाम क्या है?

English : I am learning machine learning.
Hindi   : मैं मशीन लर्निंग सीख रहा हूँ।

English : I love programming.
Hindi   : मुझे प्रोग्रामिंग बहुत पसंद है।

English : Where do you live?
Hindi   : आप कहाँ रहते हैं?


In [15]:
!pip install -q sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.8 MB/s eta 0:00:00


In [16]:
from torch.utils.data import DataLoader

test_dataset = NPZTokenizedDataset(test_chunk_files)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

print("Test dataset ready.")

Test dataset ready.


In [17]:
# ==========================================
# BLEU + chrF EVALUATION
# ==========================================

import torch
import sacrebleu

model.eval()

predictions = []
references = []

MAX_TEST_SAMPLES = 1000

with torch.no_grad():

    for batch in test_loader:

        batch = {
            k: v.to(device, non_blocking=True)
            for k, v in batch.items()
        }

        # Generate Hindi translations
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            generated_tokens = model.generate(
                input_ids=batch["encoder_inputs"],
                attention_mask=(batch["encoder_inputs"] != tokenizer.pad_token_id),
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
                max_length=128,
                num_beams=4
            )

        # Decode predictions
        pred_texts = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True
        )

        # Decode references
        ref_texts = tokenizer.batch_decode(
            batch["decoder_targets"],
            skip_special_tokens=True
        )

        predictions.extend(pred_texts)
        references.extend(ref_texts)

        if len(predictions) >= MAX_TEST_SAMPLES:
            break

# Limit exactly to requested number
predictions = predictions[:MAX_TEST_SAMPLES]
references = references[:MAX_TEST_SAMPLES]

# ==========================================
# SACREBLEU
# ==========================================

bleu = sacrebleu.corpus_bleu(
    predictions,
    [references]
)

chrf = sacrebleu.corpus_chrf(
    predictions,
    [references]
)

print("\n" + "=" * 50)
print("TRANSLATION EVALUATION")
print("=" * 50)

print(f"Test Samples : {len(predictions)}")
print(f"BLEU Score   : {bleu.score:.4f}")
print(f"chrF Score   : {chrf.score:.4f}")

print("=" * 50)


TRANSLATION EVALUATION
Test Samples : 1000
BLEU Score   : 0.0172
chrF Score   : 1.2785


In [18]:
# ==========================================
# SAVE FINAL MODEL - EPOCH 5
# ==========================================

FINAL_DIR = "/kaggle/working/models/nllb_final"

os.makedirs(FINAL_DIR, exist_ok=True)

# Save model
model.save_pretrained(FINAL_DIR)

# Save tokenizer
tokenizer.save_pretrained(FINAL_DIR)

print("=" * 60)
print("FINAL MODEL SAVED SUCCESSFULLY")
print("=" * 60)
print("Location:", FINAL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FINAL MODEL SAVED SUCCESSFULLY
Location: /kaggle/working/models/nllb_final


In [19]:
import os

print("\nSaved files:")

for file in sorted(os.listdir(FINAL_DIR)):
    print(file)


Saved files:
config.json
generation_config.json
model.safetensors
tokenizer.json
tokenizer_config.json


In [20]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

FINAL_DIR = "/kaggle/working/models/nllb_final"

final_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_DIR,
    src_lang="eng_Latn"
)

final_model = AutoModelForSeq2SeqLM.from_pretrained(
    FINAL_DIR
).to(device)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Model loaded successfully!


In [21]:
text = "i love python"

inputs = final_tokenizer(
    text,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output = final_model.generate(
        **inputs,
        forced_bos_token_id=final_tokenizer.convert_tokens_to_ids("hin_Deva"),
        max_length=128,
        num_beams=4
    )

translation = final_tokenizer.batch_decode(
    output,
    skip_special_tokens=True
)[0]

print("English:", text)
print("Hindi:", translation)

English: i love python
Hindi: मैं पाइथन से प्यार करता हूँ


In [22]:
import shutil
import os

FINAL_DIR = "/kaggle/working/models/nllb_final"
ZIP_PATH = "/kaggle/working/nllb_final"

shutil.make_archive(
    ZIP_PATH,
    "zip",
    FINAL_DIR
)

print("Model ZIP created:")
print("/kaggle/working/nllb_final.zip")

Model ZIP created:
/kaggle/working/nllb_final.zip


In [23]:
import os

FINAL_DIR = "/kaggle/working/models/nllb_final"

print("Files in nllb_final:")
for file in os.listdir(FINAL_DIR):
    print("-", file)

Files in nllb_final:
- config.json
- tokenizer.json
- model.safetensors
- generation_config.json
- tokenizer_config.json


In [24]:
import os

for root, dirs, files in os.walk(FINAL_DIR):
    for file in files:
        path = os.path.join(root, file)
        print(f"{file:25s} {os.path.getsize(path) / (1024**3):.2f} GB")

config.json               0.00 GB
tokenizer.json            0.03 GB
model.safetensors         2.29 GB
generation_config.json    0.00 GB
tokenizer_config.json     0.00 GB


In [25]:
import os
import shutil

SOURCE = "/kaggle/working/models/nllb_final"
OUTPUT = "/kaggle/working/nllb_final"

if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)

shutil.copytree(SOURCE, OUTPUT)

print("Copied successfully!")
print("Location:", OUTPUT)

Copied successfully!
Location: /kaggle/working/nllb_final


In [26]:
for root, dirs, files in os.walk(OUTPUT):
    for file in files:
        print(os.path.join(root, file))

/kaggle/working/nllb_final/config.json
/kaggle/working/nllb_final/tokenizer.json
/kaggle/working/nllb_final/model.safetensors
/kaggle/working/nllb_final/generation_config.json
/kaggle/working/nllb_final/tokenizer_config.json
